# Fetching data at warp speed

In [ ]:
# 1. Install dependencies
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

# --- CONFIGURATION ---
# NOTE: If you want to keep files permanently, mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = Path("/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["AUD/USD", "NZD/USD"]
MONTHS = [
    "202101", "202102", "202103", "202104", "202105", "202106",
    "202107", "202108", "202109", "202110", "202111", "202112",
    "202201", "202202", "202203", "202204", "202205", "202206",
    "202207", "202208", "202209", "202210", "202211", "202212"]

# PARALLELISM: Use -1 to use all cores, or 4-8 to avoid getting rate-limited
N_JOBS = 4 

# --- HELPER FUNCTIONS ---
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """Function to be executed in parallel with existence check and retries"""
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"⏭️ Skipped (Already Exists): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"⚠️ Empty: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"✅ Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"❌ FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"

# --- EXECUTION ---
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"🚀 Starting Parallel Download of {len(jobs)} jobs using {N_JOBS} workers...\n")

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

for r in results:
    print(r)

elapsed = time.time() - t0
print(f"\n✨ All done! Finished in {elapsed:.1f}s")
print(f"📁 Files saved to: {OUT_DIR.absolute()}")

Mounted at /content/drive
🚀 Starting Parallel Download of 48 jobs using 4 workers...



INFO:DUKASCRIPT:current timestamp :2023-03-01T07:30:05.662000
INFO:DUKASCRIPT:current timestamp :2023-02-01T09:30:50.665000
INFO:DUKASCRIPT:current timestamp :2023-04-03T06:40:15.778000
INFO:DUKASCRIPT:current timestamp :2023-01-02T23:30:10.876000
INFO:DUKASCRIPT:current timestamp :2023-03-01T12:02:32.550000
INFO:DUKASCRIPT:current timestamp :2023-01-03T05:30:56.088000
INFO:DUKASCRIPT:current timestamp :2023-04-03T14:03:03.380000
INFO:DUKASCRIPT:current timestamp :2023-02-01T15:13:00.244000
INFO:DUKASCRIPT:current timestamp :2023-01-03T09:14:02.095000
INFO:DUKASCRIPT:current timestamp :2023-03-01T15:24:42.759000
INFO:DUKASCRIPT:current timestamp :2023-04-04T00:43:03.777000
INFO:DUKASCRIPT:current timestamp :2023-02-01T19:18:26.552000
INFO:DUKASCRIPT:current timestamp :2023-01-03T13:46:58.359000
INFO:DUKASCRIPT:current timestamp :2023-04-04T09:39:44.479000
INFO:DUKASCRIPT:current timestamp :2023-03-01T20:53:05.007000
INFO:DUKASCRIPT:current timestamp :2023-02-01T20:30:08.160000
INFO:DUK

✅ Success: AUD/USD 202301 -> audusd_dukascopy_bid_202301.parquet (2,684,103 rows) & audusd_dukascopy_ask_202301.parquet (2,684,103 rows)
✅ Success: AUD/USD 202302 -> audusd_dukascopy_bid_202302.parquet (2,289,020 rows) & audusd_dukascopy_ask_202302.parquet (2,289,020 rows)
✅ Success: AUD/USD 202303 -> audusd_dukascopy_bid_202303.parquet (2,613,491 rows) & audusd_dukascopy_ask_202303.parquet (2,613,491 rows)
✅ Success: AUD/USD 202304 -> audusd_dukascopy_bid_202304.parquet (1,263,310 rows) & audusd_dukascopy_ask_202304.parquet (1,263,310 rows)
✅ Success: AUD/USD 202305 -> audusd_dukascopy_bid_202305.parquet (1,254,963 rows) & audusd_dukascopy_ask_202305.parquet (1,254,963 rows)
✅ Success: AUD/USD 202306 -> audusd_dukascopy_bid_202306.parquet (1,240,877 rows) & audusd_dukascopy_ask_202306.parquet (1,240,877 rows)
✅ Success: AUD/USD 202307 -> audusd_dukascopy_bid_202307.parquet (1,445,864 rows) & audusd_dukascopy_ask_202307.parquet (1,445,864 rows)
✅ Success: AUD/USD 202308 -> audusd_dukas